In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "gemma"  # Make sure Mistral is pulled using `ollama pull mistral`

def generate_story(topic):
    prompt = f"""Write an engaging and educational story about {topic} for beginners. 
Use simple and clear language to explain basic concepts. 
Include interesting facts and keep it friendly and encouraging. 
The story should be around 200-300 words and end with a brief summary of what we learned. 
Make it perfect for someone just starting to learn about this topic."""

    # Send the prompt to the local Ollama model
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False  # Use True if you want streaming (manual parsing required)
        }
    )

    # Error handling
    if response.status_code != 200:
        raise RuntimeError(f"Failed to generate story: {response.status_code} {response.text}")

    # Return generated text
    return response.json().get("response", "[No response]")

# Example usage

story = generate_story(topic)

Generated Story:
 ## The Magical Transformation of a Butterfly

Have you ever wondered how a tiny, fluttery butterfly can transform from a plump, munching caterpillar to a vibrant winged creature? This incredible process is called metamorphosis, and it's one of the most fascinating things about butterflies.

The journey starts with an egg, laid by a female butterfly on a plant that her young will munch on. From this tiny speck emerges a hungry caterpillar, who munches day and night, growing bigger and bigger. But soon, the caterpillar's appetite changes. Feeling a little weary, it finds a safe spot to rest and spin a silky cocoon around itself.

Within the cozy cocoon, something magical happens. The caterpillar's body undergoes a complete transformation. Its legs, head, and even its heart are replaced by new ones. Gradually, a butterfly emerges from the cocoon, its wings still soft and crumpled. But don't worry, they'll soon harden and become vibrant and colorful.

The newly emerged bu

In [ ]:
from gtts import gTTS
from IPython.display import Audio
import io

# Initialize text-to-speech with the generated story
tts = gTTS(story)

# Save the audio to a bytes buffer in memory
audio_bytes = io.BytesIO()
tts.write_to_fp(audio_bytes)
audio_bytes.seek(0)

# Create and display an audio player widget in the notebook
Audio(audio_bytes.read(), autoplay=False)

tts.save("../data/generated_story.mp3")


In [13]:
from transformers import VitsModel, AutoTokenizer
import torch
import numpy as np
from scipy.io.wavfile import write

# Load model and tokenizer
model = VitsModel.from_pretrained("facebook/mms-tts-eng")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")

# Tokenize and generate waveform
text = story
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform  # shape: [1, samples]

# Normalize and convert to 16-bit PCM
scaled = np.int16(output.squeeze().numpy() * 32767)

# Save as WAV file
write("../data/techno.wav", rate=16000, data=scaled)
